# MVP02 — Carga Staging

Os 9 arquivos CSV foram baixados do Kaggle (`https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data`) e
carregados manualmente no volume `mvp.staging.kaggle_raw` através do Databricks Free Edition Catalog Explorer.

In [0]:
VOLUME = "/Volumes/mvp/staging/kaggle_raw"
#Definir o parametro de volume

In [0]:
# Inventario do archivos
import pandas as pd
archivos = dbutils.fs.ls(VOLUME)
inventario = pd.DataFrame([
    {"archivo": f.name, "Tamanho_MB": round(f.size / 1024**2, 2)}
    for f in archivos
]).sort_values("archivo")

print(f"Total de archivos: {len(inventario)}")
display(inventario)


In [0]:
# Validação de que os arquivos baixados são os esperados
ESPERADOS = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]

subidos = set(inventario["archivo"])
faltantes = [a for a in ESPERADOS if a not in subidos]
extras = [a for a in subidos if a not in ESPERADOS]

print("Arquivos esperados encontrados:", len(ESPERADOS) - len(faltantes), "/", len(ESPERADOS))
print("Faltantes:", faltantes or "nenhum")
print("Não esperados:", extras or "nenhum")

assert not faltantes, "Há arquivos faltando no volume de staging."

In [0]:
# Conheciendo as columnas das tabelas
for nome in ESPERADOS:
    df = (spark.read.format("csv")
          .option("header", True).option("inferSchema", False)
          .option("multiLine", True).option("escape", '"')
          .load(f"{VOLUME}/{nome}"))
    print(f"\n{nome}  —  {len(df.columns)} colunas")
    print("   " + ", ".join(df.columns))

In [0]:
#Revisão rápida da estrutura da tabela con maior columnas
df_amostra = (spark.read.format("csv")
              .option("header", True).option("multiLine", True).option("escape", '"')
              .load(f"{VOLUME}/olist_products_dataset.csv"))
display(df_amostra.limit(10))

In [0]:
#Revisão rápida da estrutura da 2da tabela con maior columnas
df_amostra = (spark.read.format("csv")
              .option("header", True).option("multiLine", True).option("escape", '"')
              .load(f"{VOLUME}/olist_orders_dataset.csv"))
display(df_amostra.limit(10))

## Resultado MVP02

Os 9 arquivos estão disponíveis em `/Volumes/mvp/staging/kaggle_raw`, íntegros e sem transformação; o volume pertence ao Unity Catalog.